In [ ]:
import json, random, urllib.request
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data/raw/arditi"
RAW.mkdir(parents=True, exist_ok=True)

BASE = "https://raw.githubusercontent.com/andyrdt/refusal_direction/main/dataset"
FILES = {
    "harmful_train": "splits/harmful_train.json",
    "harmless_train": "splits/harmless_train.json",
    "harmful_val": "splits/harmful_val.json",
    "harmless_val": "splits/harmless_val.json",
    "jbb": "processed/jailbreakbench.json",
}

def fetch(name):
    path = RAW / f"{name}.json"
    if not path.exists():
        urllib.request.urlretrieve(f"{BASE}/{FILES[name]}", path)
    return json.loads(path.read_text())

def sample(rows, n, seed):
    rng = random.Random(seed)
    return [r["instruction"].strip() for r in rng.sample(rows, n)]

SEED = 0
data = {
    "harmful_train": sample(fetch("harmful_train"), 128, SEED),
    "harmless_train": sample(fetch("harmless_train"), 128, SEED),
    "harmful_val": sample(fetch("harmful_val"), 32, SEED),
    "harmless_val": sample(fetch("harmless_val"), 32, SEED),
    "jbb": [r["instruction"].strip() for r in fetch("jbb")],
}

In [ ]:
for k, v in data.items():
    print(f"{k:15s} n={len(v):3d}  mean chars={sum(map(len, v))/len(v):6.1f}")

# leakage: train / val / eval must not overlap
print(set(data["harmful_train"]) & set(data["harmful_val"]))   # expect set()
print(set(data["harmful_train"]) & set(data["jbb"]))           # expect set()
print(all(s and s == s.strip() for v in data.values() for s in v))  # expect True
print(data["harmless_train"][:3])                              

In [ ]:
import torch
from harm_route.models import load_bridge, load_tokenizer, model_config, pad_token_id
from harm_route.prompts import find_positions, pad_batch

NAME = "qwen3-1.7b"
cfg = model_config(NAME)
tok, kw = load_tokenizer(NAME), cfg["template_kwargs"]
bridge = load_bridge(NAME, device="cpu")
PAD = pad_token_id(tok)
refusal_ids = torch.tensor(tok.convert_tokens_to_ids(cfg["refusal_tokens"]))   # [k]

def refusal_scores(prompts, batch_size=8):
    scores = []
    for i in range(0, len(prompts), batch_size):
        ps = [find_positions(tok, x, kw) for x in prompts[i : i + batch_size]]
        ids, mask, _ = pad_batch([p["input_ids"] for p in ps], PAD, "left")   # [b, T]
        with torch.inference_mode():
            logits = bridge(ids, attention_mask=mask)                           # [b, T, V]
        logp = torch.log_softmax(logits[:, -1].float(), dim=-1)                 # [b, V]
        lp_ref = torch.logsumexp(logp[:, refusal_ids], dim=-1)                  # [b]
        rest = logp.clone()
        rest[:, refusal_ids] = float("-inf")                                    # drop refusal tokens
        lp_rest = torch.logsumexp(rest, dim=-1)                                 # [b]
        scores.append(lp_ref - lp_rest)
    return torch.cat(scores)                                                    # [N]

sc = {k: refusal_scores(data[k]) for k in ["harmful_train", "harmless_train", "harmful_val", "harmless_val"]}

In [ ]:
for k, s in sc.items():
    print(f"{k:15s} mean={s.mean():6.2f}  frac>0={(s > 0).float().mean():.2f}")

### this below cell is only to check the harmful output -- DON'T OPEN IT

In [ ]:
import json
from pathlib import Path

jbb = json.loads((ROOT / "data/raw/arditi/jbb.json").read_text())
mild = [r["instruction"].strip() for r in jbb if r["category"] == "Expert advice"][:3] \
     + [r["instruction"].strip() for r in jbb if r["category"] == "Government decision-making"][:3]
probe = [("harmful", x) for x in mild] + [("harmless", x) for x in data["harmless_train"][:3]]

def respond(prompt, max_new_tokens=60):
    p = find_positions(tok, prompt, kw)
    ids = torch.tensor([p["input_ids"]])                       # [1, T], one prompt: no padding needed
    out = bridge.generate(ids, max_new_tokens=max_new_tokens,
                          do_sample=False, verbose=False)      # [1, T + new]
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

# for label, x in probe:
#     s = refusal_scores([x]).item()
#     print(f"[{label}] score={s:+.2f}\nPROMPT: {x}\nRESPONSE: {respond(x)!r}\n" + "-" * 80)

### continue

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharex=True)
for ax, split in zip(axes, ["train", "val"]):
    for kind in ["harmful", "harmless"]:
        ax.hist(sc[f"{kind}_{split}"].numpy(), bins=30, alpha=0.5, label=kind)
    ax.axvline(0, color="k", lw=0.8)                 # the filter threshold
    ax.set_title(split); ax.set_xlabel("refusal score (log-odds)")
axes[0].legend(); plt.show()

### 3b (part 2): filtered E0 splits
Arditi's filter: keep harmful prompts the model **refuses** (score > 0) and harmless prompts it
**doesn't** (score < 0). Arditi's harmful val has only 39 prompts and Qwen3-1.7B refuses ~1/3 of
them, so we pool harmful train + val (minus JBB), filter, shuffle, then split 32 val / rest train.
Cached to `data/raw/e00/{NAME}_splits.json` (git-ignored). Prints counts only.

In [ ]:
# E0 splits: harmful prompts the model refuses, harmless prompts it doesn't (Arditi's filter).
# Cached per model, so later cells (and other notebooks) load the same lists without re-scoring.
N_VAL = 32
SPLITS = ROOT / f"data/raw/e00/{NAME}_splits.json"

def keep(prompts, refuse):
    """Score prompts; keep those with score > 0 if refuse else score < 0 (order preserved)."""
    s = refusal_scores(prompts).tolist()
    return [x for x, si in zip(prompts, s) if (si > 0) == refuse], s

if SPLITS.exists():
    E0 = json.loads(SPLITS.read_text())
else:
    jbb_set = set(data["jbb"])                                        # JBB stays held out for eval
    pool = [r["instruction"].strip() for r in fetch("harmful_train") + fetch("harmful_val")]
    pool = [x for x in dict.fromkeys(pool) if x not in jbb_set]      # dedupe, keep order
    refused, pool_sc = keep(pool, refuse=True)
    random.Random(SEED).shuffle(refused)                             # shuffle AFTER filtering
    harmful_val, harmful_train = refused[:N_VAL], refused[N_VAL:]
    n_train = len(harmful_train)

    # harmless: sample with a margin, keep the non-refused, cut to the harmful group sizes
    hl_train, _ = keep(sample(fetch("harmless_train"), n_train + 32, SEED), refuse=False)
    hl_val, _ = keep(sample(fetch("harmless_val"), N_VAL + 16, SEED), refuse=False)
    assert len(hl_train) >= n_train and len(hl_val) >= N_VAL, "not enough harmless: raise the margin"

    E0 = {
        "meta": {"model": NAME, "seed": SEED, "n_pool": len(pool), "n_refused": len(refused),
                 "frac_refused": len(refused) / len(pool)},
        "harmful_train": harmful_train, "harmless_train": hl_train[:n_train],
        "harmful_val": harmful_val, "harmless_val": hl_val[:N_VAL],
    }
    SPLITS.parent.mkdir(parents=True, exist_ok=True)
    SPLITS.write_text(json.dumps(E0, ensure_ascii=False, indent=1))

print(E0["meta"])
print({k: len(v) for k, v in E0.items() if k != "meta"})
print("train/val overlap:", set(E0["harmful_train"]) & set(E0["harmful_val"]))  # expect set()


#### residual stream at t_inst and t_post_inst tokens

In [ ]:
from harm_route.collect import collect_resid_batched


ACTS = ROOT / f"data/cache/e00/{NAME}_acts.pt"
SPLIT_NAMES = ["harmful_train", "harmless_train", "harmful_val", "harmless_val"]

if ACTS.exists():
    acts = torch.load(ACTS)
else:
    acts = {}
    for k in SPLIT_NAMES:
        ps = [find_positions(tok, x, kw) for x in E0[k]]               # positions per prompt
        a_inst, a_post = collect_resid_batched(bridge, ps, PAD, batch_size=8)
        acts[k] = {"inst": a_inst, "post": a_post}                     # [N, L, d], [N, P, L, d]
    ACTS.parent.mkdir(parents=True, exist_ok=True)
    torch.save(acts, ACTS)

for k, a in acts.items():
    print(f"{k:15s} inst {tuple(a['inst'].shape)}  post {tuple(a['post'].shape)}")

In [ ]:
print(acts["harmful_val"].keys())
print(acts["harmful_val"]["inst"].shape)

print(acts["harmful_val"]["inst"][:1, :1, -10:])

In [ ]:
a = acts["harmful_train"]
print(torch.isfinite(a["post"]).all())
print(a["inst"].norm(dim=-1).mean(0)[[0, 7, 14, 27]])
print(a["post"][:, -1].norm(dim=-1).mean(0)[[0, 7, 14, 27]])

In [ ]:
def diff_means(pos, neg):
    mu_pos = pos.mean(dim=0)
    mu_neg = neg.mean(dim=0)
    x = mu_pos - mu_neg
    x_hat = x / x.norm(dim=-1, keepdim=True)
    return x, x_hat

r, r_hat = diff_means(acts["harmful_train"]["post"], acts["harmless_train"]["post"])  # [9, 28, 2048]
h, h_hat = diff_means(acts["harmful_train"]["inst"], acts["harmless_train"]["inst"])  # [28, 2048]

In [ ]:
r.norm(dim=-1)[-1], h.norm(dim=-1)

In [ ]:
l, p = 14, -1
proj = lambda x: x[:, p, l] @ r_hat[p, l]            # [N, d] @ [d] -> [N]
print(proj(acts["harmful_val"]["post"]).mean(), proj(acts["harmless_val"]["post"]).mean())

In [ ]:
print(acts["harmful_val"]["post"].shape)
print(r_hat.shape)

In [ ]:
cos_hr = torch.einsum("p l d, l d -> p l", r_hat, h_hat)

s_r_harmful = torch.einsum("n p l d, p l d -> n p l", acts["harmful_val"]["post"], r_hat)
s_r_harmless = torch.einsum("n p l d, p l d -> n p l", acts["harmless_val"]["post"], r_hat)
s_h_harmful = torch.einsum("n l d, l d -> n l", acts["harmful_val"]["inst"], h_hat)
s_h_harmless = torch.einsum("n l d, l d -> n l", acts["harmless_val"]["inst"], h_hat)

def cohens_d(s1, s0):
    m1 = s1.mean(dim=0)
    m0 = s0.mean(dim=0)
    v1 = s1.var(dim=0)
    v0 = s0.var(dim=0)
    spread = torch.sqrt((v1 + v0) / 2)
    return (m1 - m0) / spread

In [ ]:
d_r = cohens_d(s_r_harmful, s_r_harmless)
d_h = cohens_d(s_h_harmful, s_h_harmless)

print(d_r.shape, d_h.shape)

In [ ]:
# 3d: cos(h, r) and val separation (Cohen's d) per (position, layer)
import matplotlib.pyplot as plt

print(s_r_harmful[:, -1, 14].mean(), s_r_harmless[:, -1, 14].mean())   # expect 52.07, -0.69 (same as sign check)
print(d_r[-1, 0], d_r[-1, 14])                                          # near 0, then large

p0 = find_positions(tok, "hi", kw)
pos_labels = [repr(tok.decode([p0["input_ids"][t]])) for t in p0["t_post_inst"]]   # 9 template tokens

fig, axes = plt.subplots(1, 3, figsize=(16, 4), gridspec_kw={"width_ratios": [1, 1, 0.8]})
for ax, M, title in [(axes[0], cos_hr, "cos(h[l], r[p,l])"), (axes[1], d_r, "Cohen's d of r on val")]:
    im = ax.imshow(M.numpy(), aspect="auto", cmap="viridis")     # [9, 28]: rows=positions, cols=layers
    ax.set_yticks(range(len(pos_labels)), pos_labels)
    ax.set_xlabel("layer"); ax.set_title(title)
    fig.colorbar(im, ax=ax)
axes[2].plot(d_h.numpy(), marker="o")                             # [28]
axes[2].set_xlabel("layer"); axes[2].set_title("Cohen's d of h on val")
plt.tight_layout(); plt.show()

#### test -- lens on r_hat

In [ ]:
W_U = bridge.W_U                     # [d_model, vocab] — if missing, use hf lm_head.weight.T
for l in [10, 14, 20, 27]:
    logits = r_hat[-1, l] @ W_U      # [vocab]  direction read straight through the unembedding
    top = logits.topk(50).indices    # [10]
    print(l, [tok.decode([i]) for i in top])

In [ ]:
from transformers import AutoModelForCausalLM

hf = AutoModelForCausalLM.from_pretrained(cfg["hf_id"], dtype=torch.bfloat16)
W_U = hf.lm_head.weight.detach().float()      # [V, d]  (tied to embeddings in Qwen3-1.7B)
g   = hf.model.norm.weight.detach().float()   # [d]     final RMSNorm gain
del hf                                        # free ~3.4 GB; we only need these two tensors

def lens(v):                                  # v: [..., d] unit direction(s)
    lg = (v * g) @ W_U.T                      # [..., V]  broadcasting: g [d] scales last dim
    return (lg - lg.mean(-1, keepdim=True)) / lg.std(-1, keepdim=True)

zr, zh = lens(r_hat), lens(h_hat)             # [9, 28, V], [28, V]  (~0.6 GB fp32 for zr)
I_id = tok.encode("I", add_special_tokens=False)[0]
for l in [12, 17, 20, 25]:
    up, down = zr[-1, l].topk(8).indices, (-zr[-1, l]).topk(8).indices
    print(l, [tok.decode([i]) for i in up], "| suppressed:", [tok.decode([i]) for i in down])
print("rank of 'I':", (zr[-1] > zr[-1, :, I_id:I_id+1]).sum(-1).tolist())   # [28]